In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Apache Spark uses Java, so first we must install that
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-8-jdk-headless -qq

# Unpack Spark from google drive
!tar xzf /content/drive/MyDrive/spark-3.3.0-bin-hadoop3.tgz

# Set up environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "spark-3.3.0-bin-hadoop3"

# Install findspark, which helps python locate the psyspark module files
!pip install -q findspark
import findspark
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Restarting the Spark session to ensure environment variables are correctly mapped to the JVM
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession
spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [ ]:
# 1. Download the Heterogeneity Activity Recognition dataset
!wget -q https://archive.ics.uci.edu/static/public/344/heterogeneity+activity+recognition.zip -O har_hetero.zip

# 2. Unzip the main download
!unzip -q -o har_hetero.zip -d har_hetero_extracted

# 3. Unzip the nested zip files found inside the extracted folder
!unzip -q -o "/content/har_hetero_extracted/Activity recognition exp.zip" -d /content/har_hetero_extracted/data
!unzip -q -o "/content/har_hetero_extracted/Still exp.zip" -d /content/har_hetero_extracted/data

# 4. Load the dataset using PySpark with recursive lookup
har_hetero_df = spark.read\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .option('recursiveFileLookup', 'True')\
    .csv('/content/har_hetero_extracted/data/')

# Show results
har_hetero_df.printSchema()
har_hetero_df.show(20)

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)

+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|Index| Arrival_Time|      Creation_Time|                   x|                   y|                   z|User| Model|  Device|   gt|
+-----+-------------+-------------------+--------------------+--------------------+--------------------+----+------+--------+-----+
|    0|1424696633909|1424696631914042029|         0.013748169|-0.00062561035000...|        -0.023376465|   a|nexus4|nexus4_1|stand|
|    1|1424696633909|1424696631919046912|0.014816283999999999|       -0.0016937256|         -0.0

In [ ]:
har_hetero_df.printSchema()

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [ ]:
!ls -lh har_hetero.zip /content/har_hetero_extracted/

-rw-r--r-- 1 root root 785M Jun 13 14:21 har_hetero.zip

/content/har_hetero_extracted/:
total 785M
-rwx------ 1 root root 742M May 22  2023 'Activity recognition exp.zip'
drwxr-xr-x 5 root root 4.0K Jun 13 14:22  data
-rwx------ 1 root root  43M May 22  2023 'Still exp.zip'


# **TASK 2**

In [ ]:
# Exploring data to determine ingestion and preprocessing strategy
print(f"Current Partition Count: {har_hetero_df.rdd.getNumPartitions()}")
print(f"Total Row Count: {har_hetero_df.count()}")
har_hetero_df.select('x', 'y', 'z', 'gt').show(5)

Current Partition Count: 33
Total Row Count: 36349875
+--------------------+--------------------+------------+-----+
|                   x|                   y|           z|   gt|
+--------------------+--------------------+------------+-----+
|         0.013748169|-0.00062561035000...|-0.023376465|stand|
|0.014816283999999999|       -0.0016937256| -0.02230835|stand|
|           0.0158844|       -0.0016937256|-0.021240234|stand|
|         0.016952515|        -0.003829956| -0.02017212|stand|
|           0.0158844|-0.00703430180000...| -0.02017212|stand|
+--------------------+--------------------+------------+-----+
only showing top 5 rows



In [ ]:
# Code Cell 7: Data Preprocessing
from pyspark.sql.functions import col

# 1. Cast numeric columns to DoubleType as they were inferred as strings
# 2. Handle missing values by dropping rows where the target 'gt' is null
# Explanation: Numerical casting is required for scaling; removing nulls ensures model stability.
df_preprocessed = har_hetero_df.withColumn("x", col("x").cast("double")) \
    .withColumn("y", col("y").cast("double")) \
    .withColumn("z", col("z").cast("double")) \
    .dropna(subset=["gt"])

df_preprocessed.printSchema()

root
 |-- Index: string (nullable = true)
 |-- Arrival_Time: string (nullable = true)
 |-- Creation_Time: string (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)
 |-- User: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Device: string (nullable = true)
 |-- gt: string (nullable = true)



In [ ]:
# Code Cell 8: Feature Engineering Steps
# To resolve the 'Encountered null' error, we should ensure the input data is clean.

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.sql.functions import col

# 1. Clean the data: Drop rows with nulls in feature columns ('x', 'y', 'z') and target column ('gt')
# Note: We include 'gt' because StringIndexer will also fail if it encounters null labels.
# (Assuming your DataFrame is named `df`, update if necessary)
df_preprocessed = df_preprocessed.dropna(subset=['x', 'y', 'z', 'gt'])

# 2. Verification Check: Assert that no nulls remain in ['x', 'y', 'z']
null_count = df_preprocessed.filter(col("x").isNull() | col("y").isNull() | col("z").isNull()).count()
assert null_count == 0, f"Preprocessing Check Failed: Found {null_count} rows with nulls in x, y, or z."
print("Data is clean. No null values found in x, y, or z.")

# Encoding: StringIndexer converts categorical labels (gt) into numerical indices
indexer = StringIndexer(inputCol="gt", outputCol="label")

# Transformation: VectorAssembler combines features into a single vector column
# This is now guaranteed to succeed without the 'Encountered null' error.
assembler = VectorAssembler(inputCols=["x", "y", "z"], outputCol="features")

# Scaling: StandardScaler normalizes features to have unit standard deviation
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

Data is clean. No null values found in x, y, or z.


In [ ]:
# Code Cell 9: PySpark Pipeline Construction
from pyspark.ml import Pipeline

# Constructing the pipeline stage by order:
# 1. StringIndexer (Labeling)
# 2. VectorAssembler (Feature grouping)
# 3. StandardScaler (Normalization)
pipeline = Pipeline(stages=[indexer, assembler, scaler])

# Note: This failed because df_preprocessed still contains nulls in x, y, or z.
# Before running this again, please update the dropna() in the Preprocessing cell
# to include your feature columns, then re-run that cell and this one.
try:
    pipeline_model = pipeline.fit(df_preprocessed)
    data_final = pipeline_model.transform(df_preprocessed)
    data_final.select("scaledFeatures", "label").show(5)
except Exception as e:
    print("Still encountering an error. Check if your preprocessing step removed all nulls from ['x', 'y', 'z'].")

+--------------------+-----+
|      scaledFeatures|label|
+--------------------+-----+
|[0.00373789558391...|  3.0|
|[0.00402829806162...|  3.0|
|[0.00431870081122...|  3.0|
|[0.00460910328893...|  3.0|
|[0.00431870081122...|  3.0|
+--------------------+-----+
only showing top 5 rows



In [ ]:
# Code Cell 10: Data Ingestion with Partition Count
# Re-partitioning to 100 to better distribute the 36M rows across the cluster
df_ingested = har_hetero_df.repartition(100)

print(f"New Partition Count: {df_ingested.rdd.getNumPartitions()}")

New Partition Count: 100
